In [1]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai \
langchain-text-splitters python-dotenv pinecone


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
from pinecone import Pinecone
import youtube_transcript_api as yta
import os
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

In [3]:
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

print("Gemini key loaded:", gemini_api_key is not None)
print("Pinecone key loaded:", pinecone_api_key is not None)

Gemini key loaded: True
Pinecone key loaded: True


In [4]:
pc = Pinecone(api_key=pinecone_api_key)

print("Pinecone connected!")

Pinecone connected!


In [5]:
"""index_name = "youtube-ai-research"

pc.create_index_for_model(
    name=index_name,
    cloud="aws",
    region="us-east-1",
    embed={
        "model": "llama-text-embed-v2",
        "field_map": {"text": "text"}
    }
)"""


index_name = "youtube-ai-research"
index = pc.Index(index_name)

print("Index connected!")

Index connected!


In [6]:
def extract_video_id(url):
    pattern = r"(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})"
    match = re.search(pattern, url)

    if match:
        return match.group(1)

    return None

In [7]:
youtube_url = "https://www.youtube.com/watch?v=67_aMPDk2zw"

video_id = extract_video_id(youtube_url)

print("Video ID:", video_id)

Video ID: 67_aMPDk2zw


In [8]:


def get_transcript(video_id):

    ytt_api = yta.YouTubeTranscriptApi()

    transcript_list = ytt_api.fetch(
        video_id,
        languages=["en"]
    )

    return " ".join(
        snippet.text
        for snippet in transcript_list
    )

In [9]:
transcript = get_transcript(video_id)

print("Transcript length:", len(transcript))
print(transcript[:500])

Transcript length: 3626
foreign [Music] has a curious parrot called buddy buddy has a great mimicking ability and a sharp memory buddy listens to all the conversations in Peter's home and can mimic them very accurately now when he hears feeling hungry I would like to have some for this case the probability of him saying Biryani cherries or food is much higher than the words such as bicycle or book but he doesn't understand the meaning of Biryani or food or cherries the way humans do all he is doing is using statistical


In [ ]:

#video_id = "67_aMPDk2zw"

In [ ]:
"""transcript = """"""
Large Language Models, or LLMs, are a type of artificial intelligence
model designed to understand and generate human language.

LLMs are trained on large amounts of text data. During training,
the model learns patterns, relationships between words, and how
language is structured.

Models such as GPT and Gemini are examples of large language models.

LLMs can be used for many tasks including question answering,
text generation, summarization, translation, and code generation.

A key limitation of LLMs is that they can sometimes generate
incorrect or misleading information. This is why providing reliable
context can be useful when building AI applications.

Retrieval-Augmented Generation, or RAG, is a technique that combines
information retrieval with language generation. Instead of asking
the language model to answer only from its internal knowledge,
RAG first retrieves relevant information from an external knowledge
source and then provides that information to the language model.

A typical RAG system first splits documents into smaller chunks.
The chunks are converted into numerical representations called
embeddings. These embeddings are stored in a vector database.
When a user asks a question, the system searches for chunks that
are semantically similar to the question.

The retrieved chunks are then provided to the language model as
context. The language model uses this context to generate the
final answer.
"""

In [10]:
print(type(transcript))
print(len(transcript))

<class 'str'>
3626


##Step 1.b Indexing (Text Splitter)

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript])

print("Number of chunks:", len(chunks))
print(chunks[0].page_content[:500])

Number of chunks: 5
foreign [Music] has a curious parrot called buddy buddy has a great mimicking ability and a sharp memory buddy listens to all the conversations in Peter's home and can mimic them very accurately now when he hears feeling hungry I would like to have some for this case the probability of him saying Biryani cherries or food is much higher than the words such as bicycle or book but he doesn't understand the meaning of Biryani or food or cherries the way humans do all he is doing is using statistical


In [36]:
records = []

for i, chunk in enumerate(chunks):

    records.append({
    "_id": f"{video_id}-chunk-{i}",
    "text": chunk.page_content,
    "video_id": video_id,
    "chunk_number": i,
    "source": youtube_url
})

print("Number of records:", len(records))

Number of records: 5


In [37]:
index.upsert_records(
    namespace="youtube",
    records=records
)

print("Transcript chunks uploaded!")

Transcript chunks uploaded!


In [14]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=gemini_api_key,
    temperature=0.2
)

print("Gemini model ready!")

Gemini model ready!


In [15]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant answering questions about a YouTube video.

Answer the user's question using only the provided context.

If the answer is not available in the context, say:

"I don't have enough information from the video to answer that."

Context:
{context}

Question:
{question}
""")

chain = prompt | llm

print("RAG chain ready!")

RAG chain ready!


In [46]:
def ask_video(question):

    results = index.search(
        namespace="youtube",
        query={
            "inputs": {
                "text": question
            },
            "top_k": 4
        }
    )

    context = "\n\n".join(
        hit["fields"]["text"]
        for hit in results["result"]["hits"]
    )

    response = chain.invoke({
        "context": context,
        "question": question
    })

    sources = list({
    hit["fields"].get("source")
    for hit in results["result"]["hits"]
    })

    return response.text(), sources

In [47]:
answer, sources = ask_video("What is RLHF?")

print("ANSWER:")
print(answer)

print("SOURCE:")
print(sources[0])

ANSWER:
Based on the context provided, **RLHF** stands for **reinforcement learning with human feedback**. 

It is an approach used on top of statistical predictions that involves human intervention to make models (like ChatGPT) less toxic. In this process, human feedback is used to evaluate multiple answers generated by a model and indicate which ones are toxic and which ones are not.
SOURCE:
https://www.youtube.com/watch?v=67_aMPDk2zw


In [48]:
for source in sources:
    print(source)

https://www.youtube.com/watch?v=67_aMPDk2zw


In [49]:
answer, sources = ask_video("What are Large Language Models?")

print("ANSWER:")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(source)

ANSWER:
Based on the provided video context, Large Language Models (LLMs) are models that:

* **Are trained on a huge volume of data:** This includes sources like Wikipedia articles, Google News articles, online books, and other text.
* **Contain a neural network:** Inside an LLM is a neural network with trillions of parameters that can capture complex patterns and nuances in language.
* **Use statistical predictions and RLHF:** On top of statistical predictions, LLMs use an approach called Reinforcement Learning with Human Feedback (RLHF) to refine their outputs (such as making them less toxic).
* **Operate purely on training data:** They work purely based on the data they have been trained on and do not have any subjective experience, emotions, or consciousness. 

Examples of LLMs mentioned in the video include GPT-3/GPT-4 (used by ChatGPT), PaLM 2 by Google, and Llama by Meta.

SOURCES:
https://www.youtube.com/watch?v=67_aMPDk2zw


In [17]:
answer = ask_video("What are Large Language Models?")

print(answer)

Based on the provided context, Large Language Models (LLMs) are language models that:

* **Are trained on huge volumes of data:** This includes sources like Wikipedia articles, Google News articles, and online books.
* **Contain massive neural networks:** Inside an LLM is a neural network containing trillions of parameters that capture complex patterns and nuances in language.
* **Predict and complete text:** They can complete the next set of words across a wide range of topics, such as writing a poem, providing nutrition advice, or answering questions on a history subject.
* **Use RLHF:** On top of statistical predictions, LLMs use an approach called reinforcement learning with human feedback (RLHF).

Examples of LLMs mentioned in the video include GPT-3 and GPT-4 (used by ChatGPT), PaLM 2 by Google, and Llama by Meta.


In [18]:
print(ask_video("What are some tasks LLMs can perform?"))

Based on the provided context, some tasks LLMs can perform include:

* Completing the next set of words on a history subject
* Giving nutrition advice
* Writing a poem
* Predicting the next set of words for a sentence


In [19]:
print(ask_video("What is one limitation of LLMs?"))

Based on the video, one limitation of LLMs is that they do not have any subjective experience, emotions, or consciousness, and work purely based on the data they have been trained on.


In [20]:
print(ask_video("Who invented the first computer?"))

I don't have enough information from the video to answer that.
